Aspect-Based Sentiment Analysis (ABSA)

In [ ]:
# Nama : Shabrina Maharani
# NIM : 13522134
# Praktikum : 6

# Autentikasi Hugging Face untuk Penggunaan Model Gemma

In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `mihimihu` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.

# Import Library

In [ ]:
import os
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sklearn.metrics import f1_score
from typing import Tuple, Dict, List
print('Libraries imported')

Libraries imported


# Load Model

In [ ]:
MODEL_NAME = "google/gemma-3-1b-it"
IS_DECODER_ONLY = True

HF_TOKEN = os.environ.get('HF_TOKEN', None)

print("Model:", MODEL_NAME)
print("Decoder-only:", IS_DECODER_ONLY)
if HF_TOKEN is None:
    print("\nNote: HF_TOKEN not set. Public models won't need it but private ones require authentication.")

Model: google/gemma-3-1b-it
Decoder-only: True

Note: HF_TOKEN not set. Public models won't need it but private ones require authentication.


In [ ]:
import torch

try:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto"
    )

    gen = pipeline(
        'text-generation',
        model=model,
        tokenizer=tok
    )

    print('Model and pipeline loaded successfully.')

except Exception as e:
    print('Error loading model:', e)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Device set to use cuda:0


Model and pipeline loaded successfully.


In [ ]:
# 1. Zero Prompting
PROMPT_ZERO = (
    "Classify the sentiment towards the given aspect in the sentence below.\n"
    "Answer with ONE WORD only from: positive, negative, neutral, conflict."
)

# 2. Few Prompting
PROMPT_FEW = (
    "You are an ABSA assistant. Decide the sentiment towards the given aspect.\n"
    "Answer with ONE WORD only from: positive, negative, neutral, conflict.\n\n"
    "Examples:\n"
    "Sentence: \"The sushi was fresh and tasty.\" Aspect: \"sushi\" Answer: positive\n"
    "Sentence: \"The soup arrived cold and was bland.\" Aspect: \"soup\" Answer: negative\n"
    "Sentence: \"The decor is fine.\" Aspect: \"decor\" Answer: neutral\n"
    "Sentence: \"The coffee is great sometimes, but often terrible.\" Aspect: \"coffee\" Answer: conflict\n"
    "Now classify the following pair."
)

# 3. Role Prompting
PROMPT_ROLE = (
    "You are a meticulous and highly critical restaurant reviewer. "
    "Your task is to classify the sentiment (positive, negative, neutral, conflict) "
    "towards the specific aspect provided. "
    "You must answer with ONE WORD only and nothing else."
)

# 4. Step-back Prompting
PROMPT_STEP_BACK = (
    "You are an ABSA assistant. Before answering, take a step back and state the general principle "
    "for classifying sentiment for the aspect based on the sentence. "
    "Then, apply this principle to give the final answer.\n"
    "Output the final label as ONE WORD only: positive, negative, neutral, or conflict.\n"
    "Final answer (one word only):"
)

# 5. Zero-shot-COT
PROMPT_ZERO_COT = (
    "You are an ABSA assistant. Classify the sentiment towards the given aspect.\n"
    "Let's think step by step."
)

# 6. Few-shot-COT
PROMPT_FEW_COT = (
    "You are an ABSA assistant. Think step by step before deciding the label, then output the final label as ONE WORD.\n\n"
    "--- Example 1 ---\n"
    "Sentence: \"The sushi was fresh and tasty.\" Aspect: \"sushi\"\n"
    "Steps:\n"
    "1) The opinion phrases related to 'sushi' are 'fresh' and 'tasty'.\n"
    "2) Both 'fresh' and 'tasty' are positive.\n"
    "3) The final label is positive.\n"
    "Final answer (one word only): positive\n\n"
    "--- Example 2 ---\n"
    "Sentence: \"The soup arrived cold and was bland.\" Aspect: \"soup\"\n"
    "Steps:\n"
    "1) The opinion phrases related to 'soup' are 'arrived cold' and 'bland'.\n"
    "2) Both 'arrived cold' and 'bland' are negative.\n"
    "3) The final label is negative.\n"
    "Final answer (one word only): negative\n\n"
    "--- Example 3 ---\n"
    "Sentence: \"The coffee is great sometimes, but often terrible.\" Aspect: \"coffee\"\n"
    "Steps:\n"
    "1) The opinion phrases related to 'coffee' are 'great' and 'terrible'.\n"
    "2) 'great' is positive, but 'terrible' is negative. This is a conflict.\n"
    "3) The final label is conflict.\n"
    "Final answer (one word only): conflict\n\n"
    "--- Task ---\n"
)

# 7. Generate Knowledge Prompting
PROMPT_GENERATE_KNOWLEDGE = (
    "You are an ABSA assistant. First, generate one sentence of relevant knowledge about the aspect. "
    "Then, classify the sentiment (positive, negative, neutral, conflict) based on that knowledge. "
    "Answer with the sentiment label only.\n\n"
    "--- Example 1 ---\n"
    "Sentence: \"The soup arrived cold and was bland.\" Aspect: \"soup\"\n"
    "Knowledge: Soup is generally expected to be served hot and be flavorful; 'cold' and 'bland' are undesirable qualities.\n"
    "Answer: negative\n\n"
    "--- Example 2 ---\n"
    "Sentence: \"The decor is fine.\" Aspect: \"decor\"\n"
    "Knowledge: The word 'fine' in a review context typically indicates mediocrity, not strong positive or negative feelings.\n"
    "Answer: neutral\n\n"
    "--- Task ---\n"
)

# 8. Self-consistency
PROMPT_SELF_CONSISTENCY = (
    "You are an ABSA assistant. Think step by step before deciding the label.\n"
    "Steps:\n"
    "1) Extract opinion phrases related to the aspect.\n"
    "2) Determine polarity towards that aspect.\n"
    "3) Output the final label as ONE WORD from: positive, negative, neutral, conflict.\n"
    "Final answer (one word only):"
)

# 9. Tree of Thought
PROMPT_TOT_SIMULATED = (
    "You are an ABSA assistant using a Tree of Thought process.\n"
    "Task: Determine the sentiment for the given aspect.\n\n"
    "Step 1: Generate 3 distinct reasoning paths (branches) to evaluate the sentiment.\n"
    "Step 2: Evaluate each branch. State which branch is the most logical.\n"
    "Step 3: Based on the best branch, conclude the final sentiment.\n"
    "Output the final label as ONE WORD from: positive, negative, neutral, conflict.\n"
    "Final answer (one word only):"
)

# 10. Emotion Prompt
PROMPT_EMOTION = (
    "Classify the sentiment towards the given aspect. This is very important for my analysis.\n"
    "Answer with ONE WORD only from: positive, negative, neutral, conflict."
)



PROMPT_VARIANTS = {
    "zero": PROMPT_ZERO,
    "few": PROMPT_FEW,
    "role": PROMPT_ROLE,
    "step_back": PROMPT_STEP_BACK,
    "zero_cot": PROMPT_ZERO_COT,
    "few_cot": PROMPT_FEW_COT,
    "gen_knowledge": PROMPT_GENERATE_KNOWLEDGE,
    "self_consistency": PROMPT_SELF_CONSISTENCY,
    "tot_simulated": PROMPT_TOT_SIMULATED,
    "emotion" : PROMPT_EMOTION
}

LABELS = ["positive", "negative", "neutral", "conflict"]
print('Prompt variants prepared:', list(PROMPT_VARIANTS.keys()))

Prompt variants prepared: ['zero', 'few', 'role', 'step_back', 'zero_cot', 'few_cot', 'gen_knowledge', 'self_consistency', 'tot_simulated', 'emotion']


# Inference + Post-processing

In [ ]:
def generate_text(prompt: str, max_new_tokens: int = 32, do_sample: bool = False) -> str:
    if IS_DECODER_ONLY:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=do_sample, return_full_text=False)
        text = out[0].get('generated_text', '') if isinstance(out, list) and out else (out[0] if isinstance(out, list) else str(out))
        return text.strip()
    else:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=do_sample)
        text = out[0].get('generated_text', '') if isinstance(out, list) and out else str(out)
        return text.strip()

print('Generation wrapper ready')

Generation wrapper ready


# Post Processing with Normalization

In [ ]:
CANON = {
    "positive":"positive","pos":"positive","positif":"positive","+":"positive",
    "negative":"negative","neg":"negative","negatif":"negative","-":"negative",
    "neutral":"neutral","neu":"neutral","netral":"neutral","none":"neutral",
    "conflict":"conflict","mixed":"conflict","both":"conflict","conflicted":"conflict"
}
PAT = re.compile(r"\b(positive|negative|neutral|conflict|pos|neg|neu|netral|positif|negatif|mixed|both|conflicted)\b", re.I)

def normalize_label(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return "neutral"
    m = PAT.findall(text.strip().lower())
    if m:
        return CANON.get(m[-1].lower(), "neutral")
    return "neutral"

# quick tests
print(normalize_label('Answer: positive.'))
print(normalize_label('Final answer: NEG'))
print(normalize_label('{"label":"conflict"}'))


positive
negative
conflict


## Load file test

In [ ]:
import gdown
import os
import shutil

folder_url = "https://drive.google.com/drive/folders/1IPFcC892U0qUMscBX7vwie4n9mA0MOsv"
temp_folder = "temp_download_folder"
target_folder = "."

if os.path.exists(temp_folder):
    shutil.rmtree(temp_folder)

gdown.download_folder(url=folder_url, output=temp_folder, quiet=False)

file_count = 0
for root, dirs, files in os.walk(temp_folder):
    for file in files:
        source_path = os.path.join(root, file)
        dest_path = os.path.join(target_folder, file)

        try:
            shutil.move(source_path, dest_path)
            file_count += 1
        except shutil.Error as e:
            print(f"Error: {e}")

print(f"All File downloaded")

try:
    shutil.rmtree(temp_folder)
except OSError as e:
    print(f"Error {e}")

Retrieving folder contents


Processing file 1aSTJNK9SuD4fCi9Ar2TQpuQ2PR8GZOlJ fewshot_dataset.csv
Processing file 14W8_k49gIqQ2Q-jpJCtOnQla3785u6ds sample_submission.csv
Processing file 1cBaEzHjvsBJsZykyziS2DtsCV6Lqa5wi test.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1aSTJNK9SuD4fCi9Ar2TQpuQ2PR8GZOlJ
To: /content/temp_download_folder/fewshot_dataset.csv
100%|██████████| 313k/313k [00:00<00:00, 102MB/s]
Downloading...
From: https://drive.google.com/uc?id=14W8_k49gIqQ2Q-jpJCtOnQla3785u6ds
To: /content/temp_download_folder/sample_submission.csv
100%|██████████| 11.9k/11.9k [00:00<00:00, 16.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1cBaEzHjvsBJsZykyziS2DtsCV6Lqa5wi
To: /content/temp_download_folder/test.csv
100%|██████████| 114k/114k [00:00<00:00, 83.0MB/s]

All File downloaded



Download completed


In [ ]:
# Load Test Data
if not os.path.exists('test.csv'):
    print('test.csv not found in working directory. Please download the Kaggle competition data and place test.csv here.')
else:
    test_df = pd.read_csv('test.csv')
    print('Loaded test.csv with', len(test_df), 'rows.')
    display(test_df.head())

Loaded test.csv with 1000 rows.


,id,sentence,aspect
0,0,But the staff was so horrible to us.,staff
1,1,"To be completely fair, the only redeeming fact...",food
2,2,"The food is uniformly exceptional, with a very...",food
3,3,"The food is uniformly exceptional, with a very...",kitchen
4,4,"The food is uniformly exceptional, with a very...",menu


# Inference

In [ ]:
# Inference
from tqdm.auto import tqdm
import pandas as pd
import math
from collections import Counter

BATCH_SIZE = 16

# Self-Consistency Configuration
N_SAMPLES = 5 # For voting
CONSISTENCY_TEMP = 1.0
CONSISTENCY_TOP_P = 0.95
CONSISTENCY_TOP_K = 64

if 'results' not in globals():
    results = {}

for variant_name, prompt_template in PROMPT_VARIANTS.items():
    print(f"\n=== Running inference for {variant_name.upper()} prompting ===")

    # Self-consistency only
    if variant_name == "self_consistency":
        print(f"-> Using Self-Consistency method (N={N_SAMPLES} samples)...")
        preds = []
        raws = []

        progress_bar = tqdm(test_df.iterrows(), total=len(test_df), desc=f"Processing {variant_name}")

        for _, row in progress_bar:
            sent = row.get('sentence') if 'sentence' in row.index else row.get('text', '')
            aspect = row.get('aspect', '')
            user_content = f"Sentence: \"{sent}\"\nAspect: \"{aspect}\"\nAnswer:"

            messages = [
                {"role": "system", "content": prompt_template},
                {"role": "user", "content": user_content}
            ]

            chat_prompt = tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            sample_votes = []
            raw_samples = []
            for _ in range(N_SAMPLES):
                raw_output = gen(
                    chat_prompt,
                    max_new_tokens=32,
                    do_sample=True,
                    temperature=CONSISTENCY_TEMP,
                    top_p=CONSISTENCY_TOP_P,
                    top_k=CONSISTENCY_TOP_K,
                    return_full_text=False
                )
                answer_text = raw_output[0]['generated_text'].strip()
                sample_votes.append(normalize_label(answer_text))
                raw_samples.append(answer_text)

            final_vote = Counter(sample_votes).most_common(1)[0][0]
            preds.append(final_vote)
            raws.append(str(raw_samples))
    else:
        print(f"-> Using standard batching (Batch Size={BATCH_SIZE})...")
        prompt_list = []
        for _, row in test_df.iterrows():
            sent = row.get('sentence') if 'sentence' in row.index else row.get('text', '')
            aspect = row.get('aspect', '')
            user_content = f"Sentence: \"{sent}\"\nAspect: \"{aspect}\"\nAnswer:"
            messages = [
                {"role": "system", "content": prompt_template},
                {"role": "user", "content": user_content}
            ]
            text = tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            prompt_list.append(text)

        preds = []
        raws = []
        num_batches = math.ceil(len(prompt_list) / BATCH_SIZE)
        progress_bar = tqdm(range(num_batches), desc=f"Processing {variant_name}")

        for i in progress_bar:
            batch_prompts = prompt_list[i*BATCH_SIZE : (i+1)*BATCH_SIZE]

            raw_outputs = gen(
                batch_prompts,
                max_new_tokens=32,
                do_sample=False,
                return_full_text=False
            )

            for output in raw_outputs:
                raw_answer = output[0]['generated_text'].strip()
                raws.append(raw_answer)
                preds.append(normalize_label(raw_answer))

    test_df[f'sentiment_{variant_name}'] = preds
    test_df[f'raw_{variant_name}'] = raws
    results[variant_name] = preds
    print(f'-> Completed {variant_name}, sample labels:', pd.Series(preds).value_counts().to_dict())

print('\nAll variants finished.')


=== Running inference for ZERO prompting ===
-> Using standard batching (Batch Size=16)...


Processing zero:   0%|          | 0/63 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


-> Completed zero, sample labels: {'positive': 678, 'negative': 262, 'neutral': 60}

=== Running inference for FEW prompting ===
-> Using standard batching (Batch Size=16)...


Processing few:   0%|          | 0/63 [00:00<?, ?it/s]

...

-> Completed self_consistency, sample labels: {'positive': 734, 'negative': 251, 'conflict': 11, 'neutral': 4}

=== Running inference for TOT_SIMULATED prompting ===
-> Using standard batching (Batch Size=16)...


Processing tot_simulated:   0%|          | 0/63 [00:00<?, ?it/s]

-> Completed tot_simulated, sample labels: {'positive': 670, 'negative': 250, 'conflict': 42, 'neutral': 38}

=== Running inference for EMOTION prompting ===
-> Using standard batching (Batch Size=16)...


Processing emotion:   0%|          | 0/63 [00:00<?, ?it/s]

-> Completed emotion, sample labels: {'positive': 685, 'negative': 262, 'neutral': 53}

All variants finished.


# Save Submission

In [ ]:
FINAL_VARIANT = 'zero'

if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results. Make sure you ran the inference cell for this variant.')

submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('zero_submission.csv', index=False)
print('Saved zero_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved zero_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


In [ ]:
FINAL_VARIANT = 'few'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('few_submission.csv', index=False)
print('Saved few_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved few_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


In [ ]:
FINAL_VARIANT = 'role'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('role_submission.csv', index=False)
print('Saved role_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved role_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


In [ ]:
FINAL_VARIANT = 'step_back'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('step_back_submission.csv', index=False)
print('Saved step_back_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved step_back_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


In [ ]:
FINAL_VARIANT = 'zero_cot'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('zero_cot_submission.csv', index=False)
print('Saved zero_cot_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved zero_cot_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


In [ ]:
FINAL_VARIANT = 'few_cot'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('few_cot_submission.csv', index=False)
print('Saved few_cot_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved few_cot_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,neutral
4,4,positive


In [ ]:
FINAL_VARIANT = 'gen_knowledge'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('gen_knowledge_submission.csv', index=False)
print('Saved gen_knowledge_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved gen_knowledge_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


In [ ]:
FINAL_VARIANT = 'self_consistency'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('self_consistency_submission.csv', index=False)
print('Saved self_consistency_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved self_consistency_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


In [ ]:
FINAL_VARIANT = 'tot_simulated'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('tot_submission.csv', index=False)
print('Saved tot_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved tot_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


In [ ]:
FINAL_VARIANT = 'emotion'
if f'sentiment_{FINAL_VARIANT}' not in test_df.columns:
    raise RuntimeError(f'Variant {FINAL_VARIANT} not found in results.')
submission = test_df[['id', f'sentiment_{FINAL_VARIANT}']].rename(columns={f'sentiment_{FINAL_VARIANT}':'sentiment'})
submission.to_csv('emotion_submission.csv', index=False)
print('Saved emotion_submission.csv with', len(submission), 'rows.')
display(submission.head())

Saved emotion_submission.csv with 1000 rows.


,id,sentiment
0,0,negative
1,1,positive
2,2,positive
3,3,positive
4,4,positive


# Evaluation
Evaluasi dilakukan menggunakan metrik Macro F1-score yang didapatkan setelah mengumpulkan submisi kaggle karena eksperimen ini menggunakan metode prompting (bukan fine-tuning) sehingga tidak dilakukan skema train-validation split pada data latih.

In [ ]:
# if os.path.exists('fewshot_dataset.csv'):
#     true_df = pd.read_csv('fewshot_dataset.csv')
#     if 'sentiment' not in true_df.columns:
#         raise RuntimeError('fewshot_dataset.csv must contain a "sentiment" column for evaluation')
#     scores = {}
#     for variant in PROMPT_VARIANTS.keys():
#         pred_col = f'sentiment_{variant}'
#         if pred_col in test_df.columns:
#             score = f1_score(true_df['sentiment'], test_df[pred_col], average='macro')
#             scores[variant] = score
#     print('Macro F1 per variant:')
#     for k,v in scores.items():
#         print(f' - {k}: {v:.4f}')
#     best = max(scores.items(), key=lambda x: x[1])
#     print(f'\nBest variant: {best[0]} with Macro F1 = {best[1]:.4f}')
# else:
#     print('fewshot_dataset.csv not found — skipping evaluation. If you have gold labels, place fewshot_dataset.csv in the working directory.')

In [ ]:
# Report
# Quick table summarizing counts per label for each variant
summary_rows = []
for v in PROMPT_VARIANTS.keys():
    col = f'sentiment_{v}'
    if col in test_df.columns:
        counts = test_df[col].value_counts().to_dict()
    else:
        counts = {}
    summary_rows.append({'variant': v, 'positive': counts.get('positive',0), 'negative': counts.get('negative',0), 'neutral': counts.get('neutral',0), 'conflict': counts.get('conflict',0)})

summary_df = pd.DataFrame(summary_rows).set_index('variant')
print('Label counts per variant:')
display(summary_df)

# test_df.to_csv('test_with_predictions_all_variants.csv', index=False)
# print('Saved test_with_predictions_all_variants.csv')

Label counts per variant:


,positive,negative,neutral,conflict
variant,,,,
zero,678,262,60,0
few,717,128,101,54
role,569,327,100,4
step_back,736,206,30,28
zero_cot,796,181,23,0
few_cot,380,115,501,4
gen_knowledge,536,217,245,2
self_consistency,734,251,4,11
tot_simulated,670,250,38,42
